In [ ]:
import numpy as np
import optuna
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.gross.lesson3 import Exercise

# загружаем данные
digits = load_digits()
X = digits.data.astype(np.float32) / 16.0
y = digits.target
print(f"Всего образцов: {len(X)}, Признаков: {X.shape[1]}, Классов: {len(np.unique(y))}")

# делим на train/test/val
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")


# функция для обучения и оценки
def evaluate(n_neurons, lr, n_epoch, batch_size):
    rng = np.random.default_rng(42)
    model = Exercise.create_model(
        Exercise.create_linear_layer(64, n_neurons, rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(n_neurons, 10, rng),
        Exercise.create_logsoftmax_layer(),
    )
    loss_fn = Exercise.create_nll_loss()
    Exercise.train_model(model, loss_fn, X_train, y_train, lr, n_epoch, batch_size)
    val_log_probs = model.forward(X_val)
    preds = np.argmax(val_log_probs, axis=1)
    return float(np.mean(preds == y_val))


# функция для optuna
def objective(trial):
    # предлагаем гиперпараметры
    n_neurons = trial.suggest_int("n_neurons", 32, 512)
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    n_epoch = trial.suggest_int("n_epoch", 10, 100)
    batch_size = trial.suggest_int("batch_size", 8, 256)

    # обучаем и получаем точность
    acc = evaluate(n_neurons, lr, n_epoch, batch_size)

    return acc


# запускаем optuna поиск
print("Optuna поиск гиперпараметров (50 попыток)")

# создаем исследование
study = optuna.create_study()
study.optimize(objective, n_trials=50, show_progress_bar=True)

# лучшие параметры
best_params = study.best_params
best_acc = study.best_value

print(f"\nЛучшая точность на валидации: {best_acc:.4f}")
print(f"Лучшие параметры: {best_params}")

# выводим топ-5 лучших попыток
print("\nТоп-5 лучших конфигураций:")
trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values("value", ascending=False)
for i in range(min(5, len(trials_df))):
    row = trials_df.iloc[i]
    print(
        f"{i + 1}. {row['value']:.4f} | neurons={int(row['params_n_neurons'])}, "
        f"lr={row['params_lr']:.5f}, epochs={int(row['params_n_epoch'])}, "
        f"batch={int(row['params_batch_size'])}"
    )

# финальное обучение на всех данных
print("\nФинальное обучение на объединённых данных")

X_train_val = np.vstack([X_train, X_val])
y_train_val = np.hstack([y_train, y_val])

rng_final = np.random.default_rng(42)
final_model = Exercise.create_model(
    Exercise.create_linear_layer(64, int(best_params["n_neurons"]), rng_final),
    Exercise.create_relu_layer(),
    Exercise.create_linear_layer(int(best_params["n_neurons"]), 10, rng_final),
    Exercise.create_logsoftmax_layer(),
)
loss_fn = Exercise.create_nll_loss()

Exercise.train_model(
    final_model,
    loss_fn,
    X_train_val,
    y_train_val,
    float(best_params["lr"]),
    int(best_params["n_epoch"]),
    int(best_params["batch_size"]),
)

# проверка на тесте
test_log_probs = final_model.forward(X_test)
preds = np.argmax(test_log_probs, axis=1)
test_acc = float(np.mean(preds == y_test))

print(f"\nTest accuracy: {test_acc:.4f}")
print(f"Val accuracy:  {best_acc:.4f}")
print(f"Gap: {abs(test_acc - best_acc):.4f}")

# матрица ошибок
print("\nМатрица ошибок:")

cm = np.zeros((10, 10), dtype=np.int32)
for true, pred in zip(y_test, preds, strict=True):
    cm[true, pred] += 1

cm_df = pd.DataFrame(
    cm,
    index=[f"True_{i}" for i in range(10)],
    columns=[f"Pred_{i}" for i in range(10)],
)
print(cm_df)

# анализ по цифрам
print("\nАнализ по классам:")
for digit in range(10):
    total = int(cm[digit].sum())
    correct = int(cm[digit, digit])
    acc_class = correct / total if total > 0 else 0.0
    print(f"Цифра {digit}: {correct}/{total} правильно ({acc_class * 100:.1f}%)")